# Train a lightweight weapon detector — YOLO26n

**Start here:** Runtime → Change runtime type → **T4 GPU**, then run cells in order.
This notebook contains its own scripts. No dataset account, token or repository clone is needed.
It downloads public data, prepares YOLO boxes, trains a candidate, evaluates it and exports ONNX.
Google Drive can preserve checkpoints; the last cell downloads a results ZIP.

**Coverage: gun, knife, grenade.** No generic bomb or explosion detector is trained here.
The public labels are a starting point: a spot-check found missing boxes (one known bad image
is excluded), and the dataset has **no weapon-free negative images**. Review/correct annotations
and add realistic CCTV negatives before production training. 100 epochs does not guarantee accuracy.
Camera/video groups are unknown; the starter test split is not proof of generalization to new cameras.

Source: [fcakyon / ashish dataset](https://huggingface.co/datasets/fcakyon/gun-object-detection),
declared CC BY 4.0. Attribution is retained. [Ultralytics licensing](https://www.ultralytics.com/license)
also applies to the framework and model in a commercial product.
No changes are made to the application's active detector.


## 1. Unpack the included scripts
Expand the next cell to inspect the embedded, editable source files.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
KIT = Path("/content/weapon-training-kit")
KIT.mkdir(parents=True, exist_ok=True)
EMBEDDED_FILES = {
  "scripts/prepare_weapon_data.py": "\"\"\"Download pinned public COCO data and prepare an auditable three-class YOLO starter.\"\"\"\nimport argparse\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport time\nimport urllib.request\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom scripts.train_weapons import FIELDS, SPLITS, artifact_digest, audit, save_json\n\nREVISION = '0c8e46cbfe8edf71e592f495face94ba22155b46'\nSOURCE = 'https://huggingface.co/datasets/fcakyon/gun-object-detection'\nBASE = f'{SOURCE}/resolve/{REVISION}/'\nARCHIVES = {\n    'train': ('b411ecb8ca9f4d54f3a1fb1899db14832ccef2f3c44c910ffaa1f2649d87fe79', 73680397),\n    'valid': ('d676264a3e040a71eb58ea71a4cd16391537c9eef84531df3d198311a1d86723', 18650193),\n}\nWEIGHTS_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt'\nWEIGHTS_SHA = '9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef'\nNAMES = ['gun', 'knife', 'grenade']\nMAPPING = {'pistol': 0, 'rifle': 0, 'knife': 1, 'grenade': 2}\nKNOWN_INCOMPLETE = {'00077a3cc51da6fab9ad7c78dd575e1a95fc54cfbeb8cac99d9ddff2904cef8d'}\nLIMITATIONS = [\n    'Public starter labels are upstream annotations, not human-reviewed by this project.',\n    'A visual spot-check found an unlabeled rifle beside a labeled grenade. That known image is excluded; other missing or incorrect boxes may remain.',\n    'Original camera/video identifiers are unavailable. Filename families and exact pixels are grouped; near-duplicate or scene leakage can remain.',\n    'The published validation set is reserved as test; validation is a deterministic 15% holdout from published training families.',\n    'Only gun, knife and grenade are covered. No explosion or generic bomb class.',\n    'Source images were stretched to 416x416; larger training input cannot recover lost detail. CCTV quality and false-alarm performance are unproven.',\n]\n\n\ndef download(url, destination, checksum=None, max_bytes=100_000_000):\n    \"\"\"Stream to a temporary file, retry, verify, then publish; never execute remote code.\"\"\"\n    destination = Path(destination)\n    if destination.is_file():\n        if checksum and artifact_digest(destination) != checksum:\n            raise ValueError(f'Cached checksum mismatch: {destination}; move it aside and retry.')\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + '.part')\n    for attempt in range(3):\n        try:\n            request = urllib.request.Request(url, headers={'User-Agent': 'VDM-weapon-training/1.0'})\n            size = 0\n            with urllib.request.urlopen(request, timeout=60) as source, temporary.open('wb') as target:\n                while chunk := source.read(1024 * 1024):\n                    size += len(chunk)\n                    if size > max_bytes:\n                        raise ValueError(f'Download exceeded size limit: {destination.name}')\n                    target.write(chunk)\n            if checksum and artifact_digest(temporary) != checksum:\n                raise ValueError(f'Download checksum mismatch: {destination.name}')\n            temporary.replace(destination)\n            print(f'Downloaded {destination.name}: {size / 1e6:.1f} MB', flush=True)\n            return destination\n        except (OSError, ValueError):\n            temporary.unlink(missing_ok=True)\n            if attempt == 2:\n                raise\n            time.sleep(attempt + 1)\n\n\ndef download_source(cache):\n    cache = Path(cache)\n    for split, (checksum, size) in ARCHIVES.items():\n        download(BASE + f'data/{split}.zip', cache / f'{split}.zip', checksum, size)\n    for name in ('README.md', 'README.dataset.txt', 'README.roboflow.txt'):\n        download(BASE + name, cache / name, max_bytes=1_000_000)\n\n\ndef inspect_zip(archive):\n    infos = archive.infolist()\n    if len(infos) > 20_000 or sum(info.file_size for info in infos) > 2_000_000_000:\n        raise ValueError('Archive is larger than the supported starter dataset.')\n    seen = set()\n    for info in infos:\n        path = PurePosixPath(info.filename)\n        if path.is_absolute() or '..' in path.parts or '\\\\' in info.filename or ':' in info.filename:\n            raise ValueError('Unsafe archive path: ' + info.filename)\n        if info.filename in seen or info.file_size > 50_000_000:\n            raise ValueError('Duplicate member or oversized file in archive.')\n        seen.add(info.filename)\n    candidates = [name for name in seen if name.endswith('_annotations.coco.json')]\n    if len(candidates) != 1:\n        raise ValueError('Expected exactly one _annotations.coco.json in each archive.')\n    return candidates[0]\n\n\ndef yolo_box(annotation, width, height, category_map):\n    if annotation.get('iscrowd', 0):\n        raise ValueError('Crowd annotations need manual review.')\n    category = category_map[annotation['category_id']]\n    x, y, w, h = map(float, annotation['bbox'])\n    if not all(math.isfinite(value) for value in (x, y, w, h)) or w <= 0 or h <= 0:\n        raise ValueError('Invalid COCO bounding box.')\n    # Some upstream boxes slightly cross image edges. Clip the visible extent and record it.\n    x1, y1, x2, y2 = max(0., x), max(0., y), min(float(width), x+w), min(float(height), y+h)\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError('COCO box lies outside the image.')\n    clipped = (x1, y1, x2, y2) != (x, y, x+w, y+h)\n    row = f'{category} {(x1+x2)/(2*width):.9f} {(y1+y2)/(2*height):.9f} {(x2-x1)/width:.9f} {(y2-y1)/height:.9f}'\n    return row, clipped\n\n\ndef prepare(cache, output, seed=42):\n    import cv2\n    import numpy as np\n    import yaml\n    cache, output = Path(cache), Path(output).resolve()\n    # Fail before creating output when an archive is missing, incomplete, or tampered with.\n    for split, (checksum, _) in ARCHIVES.items():\n        path = cache / f'{split}.zip'\n        if not path.is_file() or artifact_digest(path) != checksum:\n            raise ValueError(f'Missing or incorrect pinned archive: {path}')\n    output.mkdir(parents=True, exist_ok=False)\n    for split in SPLITS:\n        for kind in ('images', 'labels'):\n            (output / kind / split).mkdir(parents=True)\n    records, skipped, pixels, family_splits = [], [], {}, {}\n    stats = Counter()\n    # Held-out images take precedence over copies in the upstream training archive.\n    for original_split in ('valid', 'train'):\n        with zipfile.ZipFile(cache / f'{original_split}.zip') as archive:\n            annotation_file = inspect_zip(archive)\n            coco = json.loads(archive.read(annotation_file))\n            categories = {item['id']: MAPPING[item['name'].lower()] for item in coco['categories']}\n            by_image = defaultdict(list)\n            image_ids = {item['id'] for item in coco['images']}\n            if len(image_ids) != len(coco['images']):\n                raise ValueError('Duplicate COCO image IDs.')\n            for annotation in coco['annotations']:\n                if annotation['image_id'] not in image_ids or annotation['category_id'] not in categories:\n                    raise ValueError('Annotation references an unknown image/category.')\n                by_image[annotation['image_id']].append(annotation)\n            for item in sorted(coco['images'], key=lambda item: item['file_name']):\n                filename = item['file_name']\n                member = str(PurePosixPath(annotation_file).parent / filename)\n                suffix = PurePosixPath(filename).suffix.lower()\n                if suffix not in ('.jpg', '.jpeg', '.png', '.webp'):\n                    raise ValueError(f'Unsupported image extension: {filename}')\n                encoded = archive.read(member)\n                frame = cv2.imdecode(np.frombuffer(encoded, dtype=np.uint8), cv2.IMREAD_COLOR)\n                if frame is None or frame.shape[:2] != (item['height'], item['width']):\n                    raise ValueError(f'COCO/image dimensions disagree: {filename}')\n                digest = hashlib.sha256(str(frame.shape).encode() + frame.tobytes()).hexdigest()\n                if digest in KNOWN_INCOMPLETE:\n                    skipped.append(dict(file=filename, upstream_split=original_split, reason='known_incomplete_annotation'))\n                    stats['known_incomplete_annotation'] += 1\n                    continue\n                # Roboflow .rf.<hash> variants of one original belong to one family.\n                family = PurePosixPath(filename).name.split('.rf.')[0]\n                group = hashlib.sha256(family.encode()).hexdigest()\n                held_out = int(hashlib.sha256(f'{seed}:{family}'.encode()).hexdigest(), 16) / 2**256 < .15\n                split = 'test' if original_split == 'valid' else ('val' if held_out else 'train')\n                reason = 'duplicate_pixels' if digest in pixels else None\n                if group in family_splits and family_splits[group] != split:\n                    reason = 'family_in_heldout_split'\n                if reason:\n                    skipped.append(dict(file=filename, upstream_split=original_split, reason=reason))\n                    stats[reason] += 1\n                    continue\n                rows = []\n                for annotation in by_image[item['id']]:\n                    row, clipped = yolo_box(annotation, item['width'], item['height'], categories)\n                    rows.append(row)\n                    stats['clipped_boxes'] += int(clipped)\n                rows = sorted(set(rows))\n                key = f'images/{split}/{digest}{suffix}'\n                (output / key).write_bytes(encoded)\n                (output / 'labels' / split / f'{digest}.txt').write_text('\\n'.join(rows) + ('\\n' if rows else ''))\n                records.append(dict(image=key, source_group='filename-family:' + group,\n                    origin=f'{SOURCE}/tree/{REVISION}#{original_split}/{filename}',\n                    usage_rights='CC BY 4.0 declared by upstream; see attribution/', reviewed='upstream'))\n                pixels[digest], family_splits[group] = key, split\n                stats[f'{split}_images'] += 1\n    with (output / 'sources.csv').open('w', newline='') as stream:\n        writer = csv.DictWriter(stream, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows(records)\n    config = output / 'dataset.yaml'\n    config.write_text(yaml.safe_dump(dict(path='.', **{split: f'images/{split}' for split in SPLITS},\n                                         names=dict(enumerate(NAMES))), sort_keys=False))\n    attribution = output / 'attribution'\n    attribution.mkdir()\n    for name in ('README.md', 'README.dataset.txt', 'README.roboflow.txt'):\n        if (cache / name).is_file():\n            (attribution / name).write_bytes((cache / name).read_bytes())\n    (attribution / 'CHANGES.txt').write_text(\n        f'Source: {SOURCE}\\nRevision: {REVISION}\\nOriginal creator: ashish (Roboflow); mirror: fcakyon.\\n'\n        'Declared license: CC BY 4.0 \u2014 https://creativecommons.org/licenses/by/4.0/\\n'\n        'Changes: COCO to YOLO; pistol/rifle merged as gun; known incomplete sample excluded; edge boxes clipped; exact decoded duplicates removed; '\n        'filename families separated; validation carved from training; original validation reserved as test.\\n')\n    save_json(output / 'preparation.json', dict(source=SOURCE, revision=REVISION, archive_sha256=ARCHIVES,\n        seed=seed, names=NAMES, statistics=dict(stats), skipped=skipped, limitations=LIMITATIONS))\n    report = audit(config, allow_upstream=True)\n    save_json(output / 'dataset-audit.json', report)\n    print(json.dumps(dict(valid=report['valid'], counts=report['counts'], statistics=dict(stats)), indent=2))\n    if not report['valid']:\n        raise ValueError('Prepared data failed checks; inspect dataset-audit.json. ' + '; '.join(report['errors'][:10]))\n    return config\n\n\ndef preview(config, destination, count=12):\n    \"\"\"Show one deterministic sample per class, then fill the contact sheet.\"\"\"\n    import cv2\n    import numpy as np\n    import yaml\n    config = Path(config).resolve()\n    data = yaml.safe_load(config.read_text())\n    root = (config.parent / data.get('path', '.')).resolve()\n    paths = sorted((root / 'images/train').glob('*'))\n    selected = []\n    for index in range(len(data['names'])):\n        for image in paths:\n            label = root / 'labels/train' / (image.stem + '.txt')\n            if image not in selected and any(row.startswith(f'{index} ') for row in label.read_text().splitlines()):\n                selected.append(image)\n                break\n    selected += [image for image in paths if image not in selected][:max(0, count-len(selected))]\n    tiles = []\n    for image in selected[:count]:\n        frame = cv2.imread(str(image))\n        height, width = frame.shape[:2]\n        for row in (root / 'labels/train' / (image.stem + '.txt')).read_text().splitlines():\n            cls, x, y, w, h = map(float, row.split())\n            p1, p2 = (int((x-w/2)*width), int((y-h/2)*height)), (int((x+w/2)*width), int((y+h/2)*height))\n            cv2.rectangle(frame, p1, p2, (0, 220, 255), 2)\n            cv2.putText(frame, data['names'][int(cls)], (p1[0], max(15, p1[1])), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 220, 255), 1)\n        tiles.append(cv2.resize(frame, (320, 320)))\n    if not tiles:\n        raise ValueError('No training images to preview.')\n    while len(tiles) % 4:\n        tiles.append(np.zeros((320, 320, 3), dtype=np.uint8))\n    sheet = np.vstack([np.hstack(tiles[index:index+4]) for index in range(0, len(tiles), 4)])\n    if not cv2.imwrite(str(destination), sheet):\n        raise OSError(f'Could not write preview: {destination}')\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--cache', type=Path, default=ROOT / 'data/weapon_downloads')\n    parser.add_argument('--output', type=Path, default=ROOT / 'data/weapon_starter')\n    parser.add_argument('--seed', type=int, default=42)\n    parser.add_argument('--offline', action='store_true', help='Use verified cached archives; do not download')\n    parser.add_argument('--weights', type=Path, help='Also download the checksum-pinned official COCO yolo26n.pt here')\n    args = parser.parse_args(argv)\n    try:\n        if args.output.exists():\n            raise ValueError('Output already exists; use it as-is or choose a new directory. No overwrite.')\n        if not args.offline:\n            download_source(args.cache)\n        if args.weights:\n            if args.offline:\n                if not args.weights.is_file() or artifact_digest(args.weights) != WEIGHTS_SHA:\n                    raise ValueError('Offline mode requires the pinned pretrained weights already present.')\n            else:\n                download(WEIGHTS_URL, args.weights, WEIGHTS_SHA, 6_000_000)\n        config = prepare(args.cache, args.output, args.seed)\n        preview(config, args.output / 'preview.jpg')\n        print(f'Ready for candidate training: {config}\\nReview preview.jpg and preparation.json before using the dataset.')\n        return 0\n    except (ValueError, KeyError, OSError, zipfile.BadZipFile) as exc:\n        parser.exit(2, f'{exc}\\n')\n\n\nif __name__ == '__main__':\n    sys.exit(main())\n",
  "scripts/train_weapons.py": "\"\"\"Prepare and train local weapon-detector candidates, without activating them.\"\"\"\nimport argparse\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\n\nROOT = Path(__file__).resolve().parents[1]\nSPLITS = ('train', 'val', 'test')\nCLASSES = ['gun', 'knife', 'grenade', 'explosion']\nIMAGE_TYPES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}\nFIELDS = ['image', 'source_group', 'origin', 'usage_rights', 'reviewed']\nFINE_TUNE_OPTIONS = dict(optimizer='AdamW', lr0=0.0001, lrf=0.1, warmup_epochs=1.0,\n                         warmup_bias_lr=0.0, mosaic=0.0)\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, default=lambda item: item.item()) + '\\n')\n\n\ndef artifact_digest(path):\n    \"\"\"Checksum a file or a deterministically ordered exported model directory.\"\"\"\n    path = Path(path)\n    digest = hashlib.sha256()\n    files = sorted(p for p in path.rglob('*') if p.is_file()) if path.is_dir() else [path]\n    for file in files:\n        if path.is_dir():\n            digest.update(file.relative_to(path).as_posix().encode() + b'\\0')\n        with file.open('rb') as stream:\n            for chunk in iter(lambda: stream.read(1024 * 1024), b''):\n                digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef initialize(directory):\n    \"\"\"Create an empty dataset; an empty label explicitly means reviewed negative.\"\"\"\n    import yaml\n    directory = Path(directory).resolve()\n    directory.mkdir(parents=True, exist_ok=True)\n    for split in SPLITS:\n        for kind in ('images', 'labels'):\n            (directory / kind / split).mkdir(parents=True, exist_ok=True)\n    config = directory / 'dataset.yaml'\n    if not config.exists():\n        config.write_text(yaml.safe_dump(dict(path='.', **{split: f'images/{split}' for split in SPLITS},\n                                             names=dict(enumerate(CLASSES))), sort_keys=False))\n    manifest = directory / 'sources.csv'\n    if not manifest.exists():\n        with manifest.open('w', newline='') as stream:\n            csv.writer(stream).writerow(FIELDS)\n    print(f'Dataset scaffold: {config}\\nExisting files are preserved. Use check to inspect readiness.')\n\n\ndef audit(config_path, imgsz=640, allow_upstream=False):\n    \"\"\"Validate local YOLO directories and source separation before model loading.\"\"\"\n    import cv2\n    import yaml\n    config_path = Path(config_path).resolve()\n    config = yaml.safe_load(config_path.read_text())\n    if not isinstance(config, dict) or 'download' in config:\n        raise ValueError('Use a local dataset YAML without a download script.')\n    names = config.get('names')\n    if isinstance(names, dict) and set(names) == set(range(len(names))):\n        names = [names[index] for index in range(len(names))]\n    if (not isinstance(names, list) or not names or any(not isinstance(name, str) or not name.strip() for name in names)\n            or len(set(names)) != len(names)):\n        raise ValueError('names must be a list or a contiguous zero-based class mapping.')\n    root = (config_path.parent / config.get('path', '.')).resolve()\n    errors, warnings, counts = [], [], {}\n    if allow_upstream:\n        warnings.append('Upstream annotations are accepted for this candidate; they have not been human-reviewed here.')\n    preparation = root / 'preparation.json'\n    if preparation.is_file():\n        warnings.extend(json.loads(preparation.read_text()).get('limitations', []))\n    records = {}\n    manifest = root / 'sources.csv'\n    if manifest.is_file():\n        with manifest.open(newline='') as stream:\n            reader = csv.DictReader(stream)\n            if not set(FIELDS).issubset(reader.fieldnames or []):\n                errors.append('sources.csv is missing required columns: ' + ', '.join(FIELDS))\n            for row in reader:\n                key = row.get('image', '')\n                if key in records:\n                    errors.append(f'Duplicate provenance row: {key}')\n                records[key] = row\n    else:\n        errors.append('Missing sources.csv with origin, source grouping, rights and annotation review records.')\n    groups, pixels, seen_images, seen_labels = {}, {}, set(), set()\n    fingerprint = hashlib.sha256(config_path.read_bytes() + (manifest.read_bytes() if manifest.is_file() else b''))\n    for split in SPLITS:\n        # A deliberately narrow layout keeps image-to-label mapping unambiguous.\n        if config.get(split) != f'images/{split}':\n            raise ValueError(f'{split} must be images/{split}; use the layout created by init.')\n        directory = root / 'images' / split\n        images = sorted(p for p in directory.rglob('*') if p.suffix.lower() in IMAGE_TYPES and p.is_file())\n        classes, negatives, small = Counter(), 0, 0\n        if not images:\n            errors.append(f'{split}: no images')\n        for image in images:\n            key = image.relative_to(root).as_posix()\n            seen_images.add(key)\n            record = records.get(key, {})\n            if any(not str(record.get(field, '')).strip() for field in FIELDS):\n                errors.append(f'{key}: incomplete provenance record')\n            if record.get('reviewed', '').lower() not in ({'yes', 'upstream'} if allow_upstream else {'yes'}):\n                errors.append(f'{key}: annotations must be reviewed (reviewed=yes)')\n            group = record.get('source_group')\n            if group:\n                if group in groups and groups[group] != split:\n                    errors.append(f'{key}: source group {group!r} leaks across splits')\n                groups[group] = split\n            frame = cv2.imread(str(image))\n            if frame is None:\n                errors.append(f'{key}: unreadable image')\n                continue\n            digest = hashlib.sha256(str(frame.shape).encode() + frame.tobytes()).hexdigest()\n            if digest in pixels:\n                errors.append(f'{key}: duplicate decoded image of {pixels[digest]}')\n            pixels[digest] = key\n            label = (root / 'labels' / split / image.relative_to(directory)).with_suffix('.txt')\n            if label in seen_labels:\n                errors.append(f'{key}: multiple images map to the same label file')\n            seen_labels.add(label)\n            if not label.is_file():\n                errors.append(f'{key}: missing label; reviewed negatives require an empty .txt file')\n                continue\n            fingerprint.update(key.encode() + digest.encode() + label.read_bytes())\n            rows = label.read_text().splitlines()\n            if not any(row.strip() for row in rows):\n                negatives += 1\n            for index, row in enumerate(rows, 1):\n                if not row.strip():\n                    continue\n                try:\n                    fields = row.split()\n                    if len(fields) != 5 or not fields[0].isdigit():\n                        raise ValueError\n                    cls = int(fields[0])\n                    x, y, w, h = map(float, fields[1:])\n                    if (not 0 <= cls < len(names) or not all(math.isfinite(v) for v in (x, y, w, h))\n                            or not 0 < w <= 1 or not 0 < h <= 1\n                            or x-w/2 < -1e-5 or x+w/2 > 1+1e-5 or y-h/2 < -1e-5 or y+h/2 > 1+1e-5):\n                        raise ValueError\n                    classes[names[cls]] += 1\n                    height, width = frame.shape[:2]\n                    small += min(w*width, h*height) * imgsz/max(width, height) < 8\n                except ValueError:\n                    errors.append(f'{label.relative_to(root)}:{index}: invalid class or normalized bounding box')\n        orphaned = set((root / 'labels' / split).rglob('*.txt')) - seen_labels\n        if orphaned:\n            errors.append(f'{split}: {len(orphaned)} annotation files have no corresponding image')\n        missing = set(names) - set(classes)\n        if missing:\n            errors.append(f'{split}: no boxes for classes: {\", \".join(sorted(missing))}')\n        if not negatives:\n            warnings.append(f'{split}: add reviewed negative scenes, especially phones, tools and harmless lookalikes')\n        if small:\n            warnings.append(f'{split}: {small} boxes have a side under 8 pixels at input size {imgsz}; inspect crops/resolution')\n        if classes and min(classes.values()) < 100:\n            warnings.append(f'{split}: fewer than 100 boxes in at least one class; counts alone do not establish quality')\n        counts[split] = dict(images=len(images), negative_images=negatives, boxes=dict(classes), tiny_boxes=int(small),\n                             source_groups=len({r.get('source_group') for k, r in records.items() if k.startswith(f'images/{split}/')}))\n    extra = set(records) - seen_images\n    if extra:\n        errors.append(f'sources.csv has {len(extra)} rows without matching images')\n    canonical = dict(path=str(root), **{split: f'images/{split}' for split in SPLITS}, names=dict(enumerate(names)))\n    return dict(valid=not errors, errors=errors, warnings=warnings, counts=counts, names=names,\n                dataset_sha256=fingerprint.hexdigest(), config=canonical)\n\n\ndef configure_runtime(output, threads):\n    os.environ['YOLO_CONFIG_DIR'] = str(output / '.runtime' / 'ultralytics')\n    os.environ['MPLCONFIGDIR'] = str(output / '.runtime' / 'matplotlib')\n    os.environ['YOLO_OFFLINE'] = 'true'\n    os.environ['YOLO_AUTOINSTALL'] = 'false'\n    Path(os.environ['YOLO_CONFIG_DIR']).mkdir(parents=True, exist_ok=True)\n    import torch\n    import cv2\n    from ultralytics import YOLO, settings\n    from ultralytics import utils\n    torch.set_num_threads(threads)\n    cv2.setNumThreads(1)\n    settings.update({'sync': False, 'weights_dir': str(ROOT / 'models')})\n    # CUDA AMP checks look up this module constant, initialized at import time.\n    # Point them at the same verified base weights downloaded by the Colab setup.\n    utils.WEIGHTS_DIR = ROOT / 'models'\n    return YOLO\n\n\ndef run(args):\n    import yaml\n    report = audit(args.data, args.imgsz, args.allow_upstream_annotations)\n    if args.command == 'check':\n        print(json.dumps(report, indent=2))\n        return 0 if report['valid'] else 2\n    if not report['valid']:\n        raise ValueError('Dataset not ready:\\n' + '\\n'.join(report['errors'][:30]))\n    weights = args.weights.resolve()\n    supported_file = weights.is_file() and weights.suffix in ('.pt', '.onnx', '.engine')\n    supported_directory = weights.is_dir() and (weights.suffix == '.mlpackage' or weights.name.endswith(('_openvino_model', '_ncnn_model')))\n    if not (supported_file or supported_directory):\n        raise ValueError('Provide a local .pt/.onnx/.engine or exported OpenVINO/NCNN/CoreML directory.')\n    if args.command in ('train', 'export') and weights.suffix != '.pt':\n        raise ValueError('Training and export require a local PyTorch .pt checkpoint.')\n    output = args.output.resolve()\n    output.mkdir(parents=True, exist_ok=False)\n    data = output / 'dataset.yaml'\n    data.write_text(yaml.safe_dump(report['config'], sort_keys=False))\n    save_json(output / 'dataset-audit.json', report)\n    recipe = dict(command=args.command, candidate_only=True, dataset_sha256=report['dataset_sha256'],\n                  input_weights=str(weights), input_weights_sha256=artifact_digest(weights),\n                  imgsz=args.imgsz, nms_free=True, device=args.device, cpu_threads=args.threads,\n                  settings={key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()})\n    if getattr(args, 'fine_tune', False):\n        recipe['fine_tune_options'] = FINE_TUNE_OPTIONS\n        recipe['optimizer_state_restored'] = False\n    save_json(output / 'run.json', recipe)\n    YOLO = configure_runtime(output, args.threads)\n    model = YOLO(str(weights), task='detect')\n    if args.command == 'train':\n        if getattr(model.model.model[-1], 'one2one_cv2', None) is None:\n            raise ValueError('This recipe targets YOLO26 with its end-to-end head; use local yolo26n.pt or yolo26s.pt.')\n        if args.fine_tune:\n            names = model.names\n            names = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n            if names != report['names']:\n                raise ValueError('Fine-tuning requires checkpoint classes in exactly the dataset class order.')\n        model.train(data=str(data), epochs=args.epochs, batch=args.batch, imgsz=args.imgsz, device=args.device,\n                    workers=args.workers, project=str(output), name='fit', exist_ok=False,\n                    seed=42, deterministic=True, patience=args.patience,\n                    close_mosaic=0 if args.fine_tune else min(10, args.epochs),\n                    save_period=args.save_period, cache=False, plots=True,\n                    amp=args.device != 'cpu', nms=False, **(FINE_TUNE_OPTIONS if args.fine_tune else {}))\n        print(f'Candidate training finished: {model.trainer.best}. Evaluate before deployment.')\n        return 0\n    actual_names = model.names\n    actual_names = [actual_names[index] for index in range(len(actual_names))] if isinstance(actual_names, dict) else list(actual_names)\n    if actual_names != report['names']:\n        raise ValueError('Checkpoint class order differs from dataset; refusing misleading evaluation/export.')\n    if args.command == 'evaluate':\n        metrics = model.val(data=str(data), split=args.split, imgsz=args.imgsz, device=args.device,\n                            batch=1, workers=0, nms=False, rect=False, conf=.001,\n                            project=str(output), name='metrics', plots=True)\n        save_json(output / 'metrics.json', dict(split=args.split, aggregate=metrics.results_dict,\n                  per_class=metrics.summary(), speed=metrics.speed,\n                  note='Detection metrics, not false alarms per camera-hour or calibrated alert confidence.'))\n        print(f'Evaluation saved: {output / \"metrics.json\"}')\n    else:\n        if getattr(model.model.model[-1], 'one2one_cv2', None) is None:\n            raise ValueError('ONNX export requires a YOLO26 end-to-end candidate.')\n        # Export beside a copy, preserving the source checkpoint and any prior exports.\n        target = output / 'candidate.pt'\n        shutil.copy2(weights, target)\n        exported = YOLO(str(target), task='detect').export(format='onnx', imgsz=args.imgsz, batch=1,\n                  dynamic=False, simplify=False, opset=17, nms=False, device='cpu')\n        artifact = Path(exported)\n        save_json(output / 'candidate.json', dict(candidate_only=True, names=report['names'], imgsz=args.imgsz,\n                  format='onnx', precision='FP32', nms_free=True, file=artifact.name,\n                  sha256=hashlib.sha256(artifact.read_bytes()).hexdigest(), dataset_sha256=report['dataset_sha256']))\n        print(f'Candidate export: {artifact}. Re-evaluate this exact ONNX artifact before deployment.')\n    return 0\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest='command', required=True)\n    init = commands.add_parser('init', help='Create an empty local dataset scaffold')\n    init.add_argument('--directory', type=Path, default=ROOT / 'data' / 'weapon_dataset')\n    for command in ('check', 'train', 'evaluate', 'export'):\n        item = commands.add_parser(command)\n        item.add_argument('--data', type=Path, required=True)\n        item.add_argument('--imgsz', type=int, default=640)\n        item.add_argument('--allow-upstream-annotations', action='store_true',\n                          help='Accept reviewed=upstream public labels for candidate experiments; not a human review claim')\n        if command != 'check':\n            item.add_argument('--weights', type=Path, default=ROOT / 'models' / 'yolo26n.pt' if command == 'train' else None,\n                              required=command != 'train')\n            item.add_argument('--output', type=Path, required=True, help='New directory for this candidate run')\n            item.add_argument('--device', default='cpu', help='cpu, mps (Apple GPU), or CUDA device such as 0')\n            item.add_argument('--threads', type=int, default=1)\n        if command == 'train':\n            item.add_argument('--epochs', type=int, default=100)\n            item.add_argument('--batch', type=int, default=8)\n            item.add_argument('--workers', type=int, default=0)\n            item.add_argument('--patience', type=int, default=20, help='Early stopping patience; 0 runs all requested epochs')\n            item.add_argument('--save-period', type=int, default=-1, help='Keep an additional checkpoint every N epochs; -1 disables')\n            item.add_argument('--fine-tune', action='store_true', help='Continue trained weights with fresh AdamW, lr=0.0001 and mosaic disabled')\n        if command == 'evaluate':\n            item.add_argument('--split', choices=('val', 'test'), default='val')\n    args = parser.parse_args(argv)\n    if args.command == 'init':\n        initialize(args.directory)\n        return 0\n    if args.imgsz < 320 or args.imgsz % 32:\n        parser.error('--imgsz must be a multiple of 32, at least 320')\n    if getattr(args, 'threads', 1) < 1 or getattr(args, 'epochs', 1) < 1 or getattr(args, 'batch', 1) < 1 or getattr(args, 'workers', 0) < 0:\n        parser.error('threads, epochs and batch must be positive; workers must be nonnegative')\n    save_period = getattr(args, 'save_period', -1)\n    if getattr(args, 'patience', 0) < 0 or save_period == 0 or save_period < -1:\n        parser.error('patience must be nonnegative; save-period must be -1 or positive')\n    try:\n        return run(args)\n    except (ValueError, OSError) as exc:\n        parser.exit(2, f'{exc}\\n')\n\n\nif __name__ == '__main__':\n    sys.exit(main())\n",
  "scripts/export_weapons.py": "\"\"\"Export a trained YOLO26 weapon candidate for a specific deployment runtime.\"\"\"\nimport argparse\nimport importlib.metadata\nimport importlib.util\nfrom pathlib import Path\nimport platform\nimport shutil\nimport sys\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom scripts.train_weapons import artifact_digest, audit, configure_runtime, save_json\n\nPROFILES = {\n    'onnx-cpu': dict(format='onnx', quantize=32, simplify=False, opset=17),\n    'openvino-fp32': dict(format='openvino', quantize=32),\n    'openvino-int8': dict(format='openvino', quantize=8),\n    'coreml-fp16': dict(format='coreml', quantize=16),\n    'tensorrt-fp16': dict(format='engine', quantize=16, simplify=False, opset=17),\n}\nDEPENDENCIES = {\n    'onnx-cpu': {'onnx': 'onnx>=1.17,<2', 'onnxruntime': 'onnxruntime>=1.20,<2'},\n    'openvino-fp32': {'openvino': 'openvino>=2025.2,<2027'},\n    'openvino-int8': {'openvino': 'openvino>=2025.2,<2027', 'nncf': 'nncf>=2.14,<4'},\n    'coreml-fp16': {'coremltools': 'coremltools>=9,<10 (with numpy<=2.3.5)'},\n    'tensorrt-fp16': {'tensorrt': 'TensorRT >=8.5 matching the target CUDA/JetPack installation', 'onnx': 'onnx>=1.17,<2'},\n}\n\n\ndef check_target(target, device):\n    if target == 'coreml-fp16' and platform.system() != 'Darwin':\n        raise ValueError('Run CoreML conversion and validation on a Mac using the downloaded best.pt checkpoint.')\n    if target == 'tensorrt-fp16':\n        import torch\n        if not device.isdigit() or not torch.cuda.is_available():\n            raise ValueError('TensorRT requires the target NVIDIA CUDA device, e.g. --device 0. Build on that device.')\n    elif device != 'cpu':\n        raise ValueError('Use --device cpu for this conversion profile; the deployment device is selected at inference.')\n    missing = [spec for module, spec in DEPENDENCIES[target].items() if importlib.util.find_spec(module) is None]\n    if missing:\n        raise ValueError('Install target dependencies first: ' + '; '.join(missing))\n    if target == 'tensorrt-fp16':\n        from packaging.version import Version\n        import tensorrt\n        if Version(tensorrt.__version__) < Version('8.5'):\n            raise ValueError('TensorRT >=8.5 is required for the NMS-free head.')\n\n\ndef export_candidate(args):\n    import yaml\n    if not args.weights.is_file() or args.weights.suffix != '.pt':\n        raise ValueError('Export needs an existing, trusted local best.pt checkpoint.')\n    if args.output.exists():\n        raise ValueError('Choose a new output directory; previous exports are preserved.')\n    check_target(args.target, args.device)\n    report = audit(args.data, args.imgsz, args.allow_upstream_annotations)\n    if not report['valid']:\n        raise ValueError('Dataset check failed: ' + '; '.join(report['errors'][:10]))\n    args.output.mkdir(parents=True)\n    YOLO = configure_runtime(args.output, args.threads)\n    checkpoint = args.output / 'candidate.pt'\n    shutil.copy2(args.weights, checkpoint)\n    model = YOLO(str(checkpoint), task='detect')\n    names = model.names\n    names = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n    if names != report['names']:\n        raise ValueError('Checkpoint classes/order differ from dataset. Train on this dataset before exporting.')\n    if getattr(model.model.model[-1], 'one2one_cv2', None) is None:\n        raise ValueError('These profiles require a YOLO26 checkpoint with its NMS-free head.')\n    options = dict(PROFILES[args.target], imgsz=args.imgsz, batch=1, dynamic=False, nms=False, device=args.device)\n    calibration = None\n    if args.target == 'openvino-int8':\n        # Both entries point at training images, even if an exporter falls back to val.\n        config = dict(report['config'])\n        config['val'] = config['train']\n        config.pop('test', None)\n        calibration = args.output / 'calibration-train-only.yaml'\n        calibration.write_text(yaml.safe_dump(config, sort_keys=False))\n        options.update(data=str(calibration), split='train', fraction=args.calibration_fraction)\n    artifact = Path(model.export(**options)).resolve()\n    if not artifact.exists():\n        raise ValueError('Exporter did not produce an artifact.')\n    versions = {}\n    for package in ('ultralytics', 'torch', 'onnx', 'onnxruntime', 'openvino', 'nncf', 'coremltools', 'tensorrt'):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    manifest = dict(candidate_only=True, target=args.target, artifact=artifact.relative_to(args.output).as_posix(),\n        artifact_sha256=artifact_digest(artifact), checkpoint_sha256=artifact_digest(args.weights), names=names,\n        dataset_sha256=report['dataset_sha256'], input_shape=[1, 3, args.imgsz, args.imgsz],\n        nms_free=True, output='xyxy pixel coordinates in letterboxed input, score, class_id; undo letterboxing for display',\n        calibration_split='train' if calibration else None, options=options, versions=versions,\n        conversion_host=platform.platform(), validation_required=True,\n        note='Measure accuracy and latency on target hardware. TensorRT engines depend on GPU and TensorRT/CUDA versions.')\n    save_json(args.output / 'candidate.json', manifest)\n    save_json(args.output / 'dataset-audit.json', report)\n    print(f'Candidate export: {artifact}\\nManifest: {args.output / \"candidate.json\"}')\n    return artifact\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--weights', type=Path, required=True)\n    parser.add_argument('--data', type=Path, required=True)\n    parser.add_argument('--output', type=Path, required=True)\n    parser.add_argument('--target', choices=PROFILES, default='onnx-cpu')\n    parser.add_argument('--device', default='cpu')\n    parser.add_argument('--imgsz', type=int, default=640)\n    parser.add_argument('--threads', type=int, default=1)\n    parser.add_argument('--calibration-fraction', type=float, default=1., help='Fraction of training images for INT8, (0,1]')\n    parser.add_argument('--allow-upstream-annotations', action='store_true')\n    args = parser.parse_args(argv)\n    args.weights, args.data, args.output = (path.resolve() for path in (args.weights, args.data, args.output))\n    if args.imgsz < 320 or args.imgsz % 32 or args.threads < 1 or not 0 < args.calibration_fraction <= 1:\n        parser.error('Use imgsz >=320 divisible by 32, positive threads, and calibration fraction in (0,1].')\n    try:\n        export_candidate(args)\n        return 0\n    except (ValueError, OSError, ImportError) as exc:\n        parser.exit(2, f'{exc}\\n')\n\n\nif __name__ == '__main__':\n    sys.exit(main())\n",
  "scripts/benchmark_weapons.py": "\"\"\"Measure candidate detector latency on one image; never an accuracy benchmark.\"\"\"\nimport argparse\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport platform\nimport sys\nimport tempfile\nimport time\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom scripts.train_weapons import configure_runtime\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--weights', type=Path, nargs='+', required=True)\n    parser.add_argument('--image', type=Path)\n    parser.add_argument('--imgsz', type=int, nargs='+', default=[480, 640])\n    parser.add_argument('--runs', type=int, default=20)\n    parser.add_argument('--threads', type=int, default=1)\n    parser.add_argument('--device', default='cpu', help='cpu, CUDA index, or intel:cpu for an OpenVINO runtime')\n    parser.add_argument('--output', type=Path)\n    args = parser.parse_args()\n    if args.runs < 1 or args.threads < 1 or any(size < 320 or size % 32 for size in args.imgsz):\n        parser.error('Use positive runs/threads and image sizes divisible by 32, at least 320.')\n    if any(not ((path.is_file() and path.suffix in ('.pt', '.onnx', '.engine')) or\n                (path.is_dir() and (path.suffix == '.mlpackage' or path.name.endswith('_openvino_model')))) for path in args.weights):\n        parser.error('Provide local PyTorch, ONNX, TensorRT, OpenVINO or CoreML artifacts.')\n    if any(path.suffix != '.pt' for path in args.weights) and len(args.imgsz) != 1:\n        parser.error('For static exports, pass one --imgsz matching the export input size.')\n    with tempfile.TemporaryDirectory(prefix='vdm-weapon-benchmark-') as temporary:\n        YOLO = configure_runtime(Path(temporary), args.threads)\n        import cv2\n        import numpy as np\n        import torch\n        image = args.image or Path(importlib.util.find_spec('ultralytics').origin).parent / 'assets' / 'bus.jpg'\n        frame = cv2.imread(str(image))\n        if frame is None:\n            parser.error('Could not read image.')\n        results = []\n        for path in args.weights:\n            model = YOLO(str(path.resolve()), task='detect')\n            for size in args.imgsz:\n                settings = dict(imgsz=size, device=args.device, rect=False, nms=False, conf=.25, max_det=30, verbose=False)\n                for _ in range(3):\n                    model.predict(frame, **settings)\n                if list(model.predictor.imgsz) != [size, size]:\n                    parser.error('Requested size differs from static export input; pass the export size.')\n                cuda = model.predictor.device.type == 'cuda'\n                if cuda:\n                    torch.cuda.synchronize(model.predictor.device)\n                timings = []\n                for _ in range(args.runs):\n                    started = time.perf_counter()\n                    model.predict(frame, **settings)\n                    if cuda:\n                        torch.cuda.synchronize(model.predictor.device)\n                    timings.append((time.perf_counter()-started)*1000)\n                results.append(dict(weights=str(path), classes=len(model.names), input=[size, size],\n                                    median_ms=round(float(np.median(timings)), 2), p95_ms=round(float(np.percentile(timings, 95)), 2)))\n        report = dict(platform=platform.platform(), pytorch_cpu_threads=args.threads, device=args.device,\n                      image=str(image), warmups=3, runs=args.runs, measurements=results,\n                      note='Includes preprocessing, inference and postprocessing through the Ultralytics adapter. Exported runtimes use their own thread defaults. One-image latency only; not accuracy or full-pipeline camera capacity.')\n        encoded = json.dumps(report, indent=2) + '\\n'\n        if args.output:\n            args.output.parent.mkdir(parents=True, exist_ok=True)\n            args.output.write_text(encoded)\n        print(encoded)\n\n\nif __name__ == '__main__':\n    main()\n",
  "requirements-weapon-training.txt": "# Standalone Colab/local training toolkit. Keep the platform's matched torch/torchvision pair.\nultralytics==8.4.154\nPyYAML>=6,<7\nonnx>=1.17,<2\nonnxruntime>=1.20,<2\n",
  "docs/weapon-colab.md": "# Colab weapon training kit\n\nUpload `notebooks/weapon_training_colab.ipynb` at https://colab.research.google.com/\n(File \u2192 Upload notebook), choose **Runtime \u2192 Change runtime type \u2192 T4 GPU**, then\nrun the cells in order. The notebook contains the Python scripts itself: no GitHub\nrepository, dataset token, or second upload is required. It installs the standalone\nrequirements, downloads data and pretrained weights, trains a YOLO26n candidate,\nevaluates it and exports ONNX. Optional Google Drive storage is enabled to preserve\ncheckpoints across Colab resets. Colab availability and session limits vary.\n\n`artifacts/weapon-training-kit.zip` contains the notebook, editable scripts and this\nguide for local use. It does **not** contain a newly trained model or dataset images.\nRun `python scripts/build_weapon_notebook.py` after editing scripts to refresh both\nthe notebook's embedded copies and ZIP.\n\n## What the downloader actually provides\n\nThe pinned [fcakyon/gun-object-detection dataset](https://huggingface.co/datasets/fcakyon/gun-object-detection)\nmirrors [ashish's Roboflow dataset](https://universe.roboflow.com/ashish-cuamw/test-y7rj3).\nIts source documentation declares **CC BY 4.0**. The script preserves attribution,\nsource revision, archive SHA-256 checksums, the declared license and conversion\nchanges. This records the publisher's declaration, not an independent verification\nof every original image's rights.\n\n- Download: two archives, about **92 MB**, containing **4,666 images**.\n- Prepared labels: **0 gun** (pistol + rifle), **1 knife**, **2 grenade**.\n- One known incomplete sample is excluded: a visible rifle lacked a box beside a\n  labeled grenade. With seed 42, this leaves **3,209 train / 551 val / 905 test**.\n- Original validation is reserved as test. Validation is a deterministic 15% split\n  of original training filename families. Exact decoded duplicates are removed;\n  variants sharing the original filename stay together. Original videos/cameras\n  are unknown, so near-duplicate and scene leakage can remain.\n- All retained source images contain annotated weapons: **there are no negative\n  images**. Add reviewed weapon-free CCTV, phones, tools and other lookalikes.\n- These are **upstream starter annotations**, not fully reviewed training labels.\n  More missing boxes may remain. Import the prepared YOLO images/labels into an\n  annotation tool, correct all target boxes, and update reviewed records as you\n  review them. The notebook's preview samples all three classes.\n- There is **no explosion or generic bomb class**. Images were stretched to 416 \u00d7\n  416 upstream; training at 640 cannot recover lost detail.\n\nThe scripts deliberately record `reviewed=upstream`, and experiments using these\nlabels require `--allow-upstream-annotations`. They never mark downloaded labels as\nhuman-reviewed. Passing the audit establishes file consistency, not accuracy.\nKeep a separate, reviewed CCTV test set for any deployment decision.\n\n## Local download, prepare and train\n\nUse a Python environment with a matching PyTorch/torchvision pair for your platform.\nColab supplies these already. Install this kit's requirements in that environment:\n\n```sh\npython -m pip install -r requirements-weapon-training.txt\npython scripts/prepare_weapon_data.py \\\n  --cache data/weapon_downloads --output data/weapon_starter \\\n  --weights models/yolo26n.pt\npython scripts/train_weapons.py check \\\n  --data data/weapon_starter/dataset.yaml --allow-upstream-annotations\npython scripts/train_weapons.py train \\\n  --data data/weapon_starter/dataset.yaml --allow-upstream-annotations \\\n  --weights models/yolo26n.pt --device 0 --imgsz 640 --batch 8 --epochs 100 \\\n  --output runs/weapons-v1\n```\n\nFor CPU training use `--device cpu --batch 2`; this is much slower. If CUDA runs out\nof memory, reduce batch to 4 or 2. The starting `yolo26n.pt` is COCO-pretrained, **not\na pretrained weapon specialist**. The downloader verifies its official release\nchecksum. Output directories must be new; nothing is silently overwritten. To\nprepare from an existing archive cache without network, pass `--offline`.\n\nFor your own annotations, use `train_weapons.py init`, fill its dataset layout and\nprovenance CSV, and follow [the full training guide](weapon-training.md). That\nscaffold defaults to four classes, including explosion; remove or add classes\ndeliberately and provide examples in every split. Do not mix that class order with\na three-class checkpoint. All images need matching labels, including empty label\nfiles for reviewed negatives. Keep all frames from one video/site together.\n\n## Convert for the deployment device\n\nStart with FP32 ONNX as the portable CPU baseline. All profiles take a trained\n`best.pt` and the matching dataset YAML; quantization may change detection quality.\n\n| Target | Script profile | Extra dependencies / execution location |\n| --- | --- | --- |\n| Windows/Linux/macOS CPU | `onnx-cpu` | Included in base requirements |\n| Intel CPU or supported Intel accelerator | `openvino-fp32` | `openvino>=2025.2,<2027` |\n| Intel CPU / supported INT8 device | `openvino-int8` | Above plus `nncf>=2.14,<4`; training data for calibration |\n| Apple Silicon / Apple deployment | `coreml-fp16` | Run on Mac; `coremltools>=9,<10`, `numpy<=2.3.5` |\n| NVIDIA GPU / Jetson | `tensorrt-fp16` | Run on target CUDA device; TensorRT >=8.5 matching CUDA/JetPack |\n\nInstall only the extra dependencies for the profile you need, ideally in a separate\nconversion environment. For CoreML, for example:\n`python -m pip install 'coremltools>=9,<10' 'numpy<=2.3.5'`.\nUse NVIDIA's target-specific TensorRT/JetPack installation instructions rather than\ninstalling arbitrary CUDA packages over the Colab or Jetson runtime.\n\n```sh\npython scripts/export_weapons.py \\\n  --weights runs/weapons-v1/fit/weights/best.pt \\\n  --data data/weapon_starter/dataset.yaml --allow-upstream-annotations \\\n  --target onnx-cpu --imgsz 640 --output runs/weapons-v1-onnx\n```\n\nSubstitute the table's profile and a new output directory. TensorRT additionally\nneeds `--device 0`. INT8 uses **training images only**, including when the exporter\ndefaults to a validation entry; a separate calibration YAML prevents accidentally\nusing held-out images. `--calibration-fraction 0.25` uses a subset; ensure it covers\nrepresentative scenes and all classes. The default uses the full training split.\n\nEvery profile exports a fixed square batch-1 model with YOLO26's NMS-free head.\n`candidate.json` records class order, input size, checksums, package versions and\ncalibration settings. The original `.pt` is preserved alongside the export.\nTensorRT engines depend on GPU/runtime versions: rebuild on the intended device.\nCoreML conversion and validation are separate Mac steps, not Colab steps. ONNX on\nARM requires a compatible ONNX Runtime build; this kit does not provide Android,\niOS app integration, TFLite, NCNN or vendor-specific NPU packages.\n\n## Measure the exported model\n\n```sh\npython scripts/train_weapons.py evaluate \\\n  --weights runs/weapons-v1-onnx/candidate.onnx \\\n  --data data/weapon_starter/dataset.yaml --allow-upstream-annotations \\\n  --split val --imgsz 640 --output runs/weapons-v1-onnx-val\npython scripts/benchmark_weapons.py \\\n  --weights runs/weapons-v1/fit/weights/best.pt runs/weapons-v1-onnx/candidate.onnx \\\n  --imgsz 640 --device cpu --runs 30 --output runs/latency.json\n```\n\nThe same evaluation command accepts OpenVINO directories, `.mlpackage` directories\non Mac, or `.engine` files with `--device 0`. Benchmark on each target computer;\nruntime thread defaults differ. The benchmark includes the Ultralytics adapter's\npre/postprocessing and inference, not the complete camera/alert pipeline. It is a\nlatency measurement, not accuracy or supported-camera-count evidence. Use validation\nto select settings, then use `--split test` once settings are fixed. Compare per-class\nrecall/AP before and after conversion. Operational acceptance also needs false alarms\nper camera-hour and small/distant-weapon misses on representative CCTV.\n\nNo trained accuracy or speed improvement is promised. The project's existing weapon\ndetector remains active; integrating a candidate, its class mapping and its optimized\nruntime is a separate change after validation. Framework/model licensing also applies\nto exported weights; check [Ultralytics' licensing options](https://www.ultralytics.com/license)\nfor the intended product distribution.\n\n## Verification in this workspace\n\nThe pinned archives were downloaded and the complete 4,665-image prepared dataset\npassed structural checks. A CPU smoke run trained one epoch on **12 real training\nimages**, exported FP32 ONNX, evaluated it on a separate six-image subset, and ran\nthe PyTorch/ONNX latency harness. These tiny subsets verify execution only; they do\nnot establish a useful trained model. Automated tests cover data conversion,\nchecksums, duplicates, class mappings, export profiles, train-only calibration and\nnotebook/bundle consistency. The Colab GPU session and native OpenVINO, CoreML and\nTensorRT conversions have **not** been executed here; validate those on their targets.\n\nImplementation references: [export API](https://docs.ultralytics.com/modes/export/),\n[OpenVINO](https://docs.ultralytics.com/integrations/openvino/),\n[CoreML](https://docs.ultralytics.com/integrations/coreml/),\n[TensorRT](https://docs.ultralytics.com/integrations/tensorrt/).\n",
  "docs/weapon-training.md": "# Lightweight weapon detector training\n\nStart with **YOLO26n**, fine-tuned from local pretrained weights. Its optional\nNMS-free head is suitable for a compact ONNX deployment. Compare YOLO26s on the same\ndata only if nano misses too many small weapons. A GPU is useful for training;\ndeployment can use CPU. Neither customer-device FPS nor weapon accuracy is guaranteed.\n\nFor a downloadable public starter dataset, a self-contained Colab notebook, and\ndevice-specific export commands, start with [the Colab kit](weapon-colab.md).\nIt covers gun, knife and grenade; its upstream annotations still need review.\nThis setup also provides dataset checks, training, evaluation and a latency harness.\n**There is no production-trained weapon model included.** The current pinned detector remains active. The dashboard's Training\npage trains the event classifier, not weapon bounding boxes.\n\n## Dataset preparation\n\n```sh\n.venv/bin/python -m pip install -e '.[vision,weapon-training,test]'\n.venv/bin/python scripts/train_weapons.py init\n```\n\nThe scaffold is `data/weapon_dataset/`, ignored by Git:\n\n```text\ndataset.yaml\nsources.csv\nimages/train/     labels/train/\nimages/val/       labels/val/\nimages/test/      labels/test/\n```\n\nThe initial names preserve the current detector's intended coverage:\n\n| ID | Class | Annotation meaning |\n| --- | --- | --- |\n| 0 | gun | Visible handgun or long gun; box the object, not the person |\n| 1 | knife | Visible knife, including its blade and handle |\n| 2 | grenade | Recognizable visible grenade-like object |\n| 3 | explosion | Visible explosion-like region; an event appearance, not a bomb object |\n\nA camera cannot establish whether a gun is real/loaded or a bag contains explosives.\nDo not infer a generic `bomb` label from suspicious bags or behavior. There is no\ncatch-all \u201cany weapon\u201d class. Add machetes, axes or batons as separately defined\nclasses only with suitable annotations; their runtime alert mappings also need\nupdating before activation. Document inert replicas and evaluate that ambiguity.\n\nUse CVAT or another annotation tool to export YOLO detection labels. Each image\nneeds a matching `.txt` file in `labels/<split>/`, with one row per object:\n\n```text\nclass_id x_center y_center width height\n```\n\nCoordinates are normalized to [0,1] using image width/height. Annotate **all target\nobjects**. An empty label file explicitly means a reviewed negative; missing files\nare rejected. Do not add a background class. Fight-clip labels are not box labels.\n\nAdd one `sources.csv` row per image:\n\n```csv\nimage,source_group,origin,usage_rights,reviewed\nimages/train/entrance_001.jpg,site-a-camera-1,owned recording reference,documented permission reference,yes\n```\n\nThe rights field records your evidence; the script cannot verify permission itself.\nKeep each camera/site group, original video, burst and augmented/re-encoded copies\nin one split. Start around 70/15/15 **by source groups**, not neighboring frames.\nValidation selects settings; reserve the test set for the final comparison. The\nchecker finds overlapping groups and identical decoded images. Near-duplicates and\nincorrect grouping still need manual review.\n\nCollect realistic CCTV views: distant objects, occlusion, low light, infrared, blur,\ncompression and diverse backgrounds. Include no-weapon footage and hard negatives\nsuch as phones, tools, umbrellas and reflections. An initial collection budget of\nroughly 1,000+ diverse annotated instances per class is a planning target, not an\naccuracy threshold. Use learning curves to decide when more data is needed.\nThousands of adjacent frames of one weapon are not thousands of independent examples.\nKeep continuous negative videos for measuring false alarms per camera-hour.\n\nPrefer owned or permissioned footage. Public datasets can seed coverage after\nchecking annotation completeness, domain match and rights. [Open Images V7](https://storage.googleapis.com/openimages/web/download_v7.html)\nprovides images and bounding boxes for many classes; it is not a complete weapon\ndataset. Preserve attribution and check its [license information](https://storage.googleapis.com/openimages/web/factsfigures_v7.html#licenses).\nHuman-review any model-generated annotations before including them.\n\n```sh\n.venv/bin/python scripts/train_weapons.py check --data data/weapon_dataset/dataset.yaml\n```\n\nTraining rejects empty splits, missing classes/labels, invalid boxes, unreviewed or\nmissing provenance, duplicates and source leakage. Warnings identify tiny boxes\nafter resize and low class counts. Passing establishes structure, not label accuracy.\n\n## Train, evaluate and export\n\n`models/yolo26n.pt` already exists here. Other machines need official local weights;\nthis script does not automatically download models or datasets. Use a **new output\ndirectory** per run. Example on a CUDA training workstation:\n\n```sh\n.venv/bin/python scripts/train_weapons.py train \\\n  --data data/weapon_dataset/dataset.yaml --weights models/yolo26n.pt \\\n  --device 0 --imgsz 640 --batch 8 --epochs 100 --output tmp/weapons-nano-v1\n```\n\nUse `--device cpu --batch 2` for a small plumbing check; CPU training can be slow.\nThe recipe uses a pretrained backbone, fixed seed, early stopping and late mosaic\nshutdown. Begin at 640 pixels: shrinking input can erase a distant knife. Compare\n480 only after checking small/distant-object recall. If testing targeted crops,\nretain full-frame scans; pose-only crops miss unattended or occluded objects.\n\n```sh\n.venv/bin/python scripts/train_weapons.py evaluate \\\n  --data data/weapon_dataset/dataset.yaml \\\n  --weights tmp/weapons-nano-v1/fit/weights/best.pt --split val \\\n  --output tmp/weapons-nano-v1-validation\n\n.venv/bin/python scripts/train_weapons.py export \\\n  --data data/weapon_dataset/dataset.yaml \\\n  --weights tmp/weapons-nano-v1/fit/weights/best.pt \\\n  --output tmp/weapons-nano-v1-onnx\n\n.venv/bin/python scripts/train_weapons.py evaluate \\\n  --data data/weapon_dataset/dataset.yaml \\\n  --weights tmp/weapons-nano-v1-onnx/candidate.onnx --split test \\\n  --output tmp/weapons-nano-v1-onnx-test\n```\n\nReports record per-class metrics, dataset fingerprint, checkpoint checksum, settings\nand export checksum. Class order must match the dataset; an older checkpoint with a\ndifferent order needs an appropriately remapped annotation copy before comparison.\nExport uses static square batch-1 FP32 ONNX with the NMS-free head. Re-evaluate the\nexported artifact; successful conversion is not proof of equivalence or accuracy.\n\n```sh\n.venv/bin/python scripts/benchmark_weapons.py \\\n  --weights models/threat-yolov8n.pt models/yolo26n.pt \\\n  --imgsz 480 640 --threads 1 --output tmp/weapon-cpu-benchmark.json\n```\n\nThis measures PyTorch CPU adapters on one identical image at fixed square sizes,\nincluding preprocessing/postprocessing, three warmups and twenty timed runs.\nCOCO-pretrained YOLO26n is not yet a trained weapon detector. Benchmark exported\nruntimes and the complete multi-camera pipeline separately on target hardware.\n\nDevelopment baseline, 2026-09-22, Apple M5/macOS arm64, one CPU thread and ten timed\nruns on bundled `bus.jpg` (median adapter time):\n\n| Model | 480 \u00d7 480 | 640 \u00d7 640 |\n| --- | ---: | ---: |\n| Existing YOLOv8n weapon checkpoint, 4 classes | 35.04 ms | 52.79 ms |\n| COCO YOLO26n checkpoint, 80 classes | 36.85 ms | 60.23 ms |\n\nYOLO26n did not beat the current detector in this PyTorch check. These different\nclass heads and training tasks do not establish a final weapon-model comparison.\nKeep the existing model as a baseline; compare the trained candidate and exported\nruntime before claiming a speed or accuracy gain. The 320-pixel, one-epoch generated\nfixture used to smoke-test train \u2192 export \u2192 evaluate is software verification only.\n\n## Deployment and acceptance\n\n- First deployment candidate: ONNX Runtime CPU.\n- On supported Intel hardware, test OpenVINO INT8 calibrated using representative\n  **training-partition** images. Re-evaluate small-weapon recall after quantization.\n- On Apple Silicon, measure CoreML; on NVIDIA hardware, measure TensorRT FP16.\n- If nano's recall is inadequate, compare a small model or experiment with a larger\n  offline teacher and distilled nano student. OpenVINO INT8 is available through\n  `export_weapons.py`; distillation is not implemented.\n\nMeasure per-class PR/AP, small/distant-object misses, event-level recall, false alarms\nper camera-hour and capture-to-alert p50/p95 under simultaneous load. The built-in\nevaluation supplies detection metrics, not operational event metrics. Select\nper-class alert thresholds on validation PR curves, then lock them for the final\ntest. A 0.9 model score does not imply 90% real-world reliability.\n\nThe current worker runs YOLO26s general objects before publishing weapon results\nand samples up to 2 FPS. Faster inference alone cannot remove that scheduling delay.\nLater integration should prioritize weapon results and measure the optional general\ncontext cost while retaining full-frame safety scans in Eco mode.\n\nNew checkpoints are **candidates**, not automatically installed: `ThreatModel` still\naccepts only its pinned checksum and known classes. Runtime integration, alert\nthreshold calibration, packaging and target-device acceptance follow evaluation.\n\nFor proprietary sales, account for framework/model licensing as well as image rights.\nUltralytics offers AGPL-3.0 and Enterprise licensing; its [published terms](https://www.ultralytics.com/license)\nspecify Enterprise for proprietary use. Fine-tuning/exporting does not remove those\nobligations.\n\nSources: [YOLO26 and head selection](https://docs.ultralytics.com/models/yolo26/),\n[YOLO box format](https://docs.ultralytics.com/datasets/detect/),\n[OpenVINO export/calibration](https://docs.ultralytics.com/integrations/openvino/).\n"
}
for name, source in EMBEDDED_FILES.items():
    destination = KIT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source)
os.chdir(KIT)
if str(KIT) not in sys.path: sys.path.insert(0, str(KIT))
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)
print("Scripts written to", KIT)


In [ ]:
run('-m', 'pip', 'install', '-q', '-r', KIT / 'requirements-weapon-training.txt')
import torch
import ultralytics
print('PyTorch:', torch.__version__, 'Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before training (Runtime → Change runtime type).'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Choose training settings and persistent storage
Start at 640 pixels and batch 8. Reduce batch to 4 or 2 if GPU memory runs out.
Save to Drive to keep `best.pt` and `last.pt` if Colab disconnects. Without Drive, download
the results before the runtime resets. Use a fresh run name for a new experiment.


In [ ]:
from datetime import datetime, timezone
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
EPOCHS = 100  # @param {type:"integer"}
IMAGE_SIZE = 640  # @param [320, 480, 640] {type:"raw"}
BATCH_SIZE = 8  # @param {type:"integer"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/vdm-weapon-training')
else:
    OUTPUT_ROOT = KIT / 'runs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = 'weapons-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
TRAIN_OUT = OUTPUT_ROOT / RUN_ID
DATA_ROOT = KIT / 'data/weapon_starter'
DATA = DATA_ROOT / 'dataset.yaml'
BASE_WEIGHTS = KIT / 'models/yolo26n.pt'
BEST = TRAIN_OUT / 'fit/weights/best.pt'
UPSTREAM = ['--allow-upstream-annotations']
print('Run:', TRAIN_OUT)


## 3. Download and prepare data
Downloads ~92 MB plus ~5.5 MB of official pretrained weights, with pinned checksums.
COCO boxes become YOLO boxes; pistol/rifle merge as **gun**. Seed 42 produces
**3,209 train / 551 validation / 905 test** images after excluding one known incomplete label.
The original validation partition becomes test; validation is held out from original training.
Exact duplicate pixels and filename families are checked, but unknown video/scene overlap can remain.


In [ ]:
if not DATA_ROOT.exists():
    run('scripts/prepare_weapon_data.py', '--cache', KIT / 'data/downloads',
        '--output', DATA_ROOT, '--weights', BASE_WEIGHTS)
else:
    from scripts.prepare_weapon_data import download, WEIGHTS_URL, WEIGHTS_SHA
    download(WEIGHTS_URL, BASE_WEIGHTS, WEIGHTS_SHA, 6_000_000)
    print('Reusing existing prepared data; auditing it again below.')
run('scripts/train_weapons.py', 'check', '--data', DATA, '--imgsz', IMAGE_SIZE, *UPSTREAM)


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(DATA_ROOT / 'preview.jpg')))
print((DATA_ROOT / 'preparation.json').read_text())


Inspect the boxes above. More missing/incorrect boxes may remain. To improve this dataset,
edit the YOLO labels in an annotation tool and add reviewed CCTV negatives, keeping videos/sites
together. The scripts retain `reviewed=upstream`; this experiment flag does not claim human review.
Re-run the audit after edits. Do not tune thresholds against the reserved test set.

## 4. Train the candidate
Fine-tunes COCO-pretrained YOLO26n with its NMS-free head. Training logs, metrics,
class order, data fingerprint and best/last checkpoints are saved under the run directory.
A one-epoch run is only a pipeline check, not a useful trained detector.


In [ ]:
run('scripts/train_weapons.py', 'train', '--data', DATA, *UPSTREAM,
    '--weights', BASE_WEIGHTS, '--output', TRAIN_OUT, '--device', '0',
    '--imgsz', IMAGE_SIZE, '--epochs', EPOCHS, '--batch', BATCH_SIZE, '--workers', '2')
assert BEST.is_file(), 'Training did not produce best.pt; inspect the training log.'
print('Candidate checkpoint:', BEST)


**After a disconnected session:** re-create the scripts/environment and prepared data first.
In a separate cell, use the saved Drive checkpoint with
`YOLO('/content/drive/MyDrive/vdm-weapon-training/<run>/fit/weights/last.pt').train(resume=True, device=0)`.
Import `YOLO` from `ultralytics` first. Resuming requires an unfinished checkpoint and the same
dataset paths/settings. Set `TRAIN_OUT` and `BEST` back to that run before executing later cells.

## 5. Validate PyTorch and export portable FP32 ONNX
Compare validation results before and after conversion. Select the input size and thresholds
using validation; reducing resolution can hide distant knives. Conversion success alone is
not a quality or speed result.


In [ ]:
PT_VAL = OUTPUT_ROOT / (RUN_ID + '-pt-val')
run('scripts/train_weapons.py', 'evaluate', '--data', DATA, *UPSTREAM,
    '--weights', BEST, '--output', PT_VAL, '--split', 'val', '--device', '0', '--imgsz', IMAGE_SIZE)
ONNX_OUT = OUTPUT_ROOT / (RUN_ID + '-onnx')
run('scripts/export_weapons.py', '--data', DATA, *UPSTREAM, '--weights', BEST,
    '--target', 'onnx-cpu', '--output', ONNX_OUT, '--imgsz', IMAGE_SIZE)
ONNX = ONNX_OUT / 'candidate.onnx'
ONNX_VAL = OUTPUT_ROOT / (RUN_ID + '-onnx-val')
run('scripts/train_weapons.py', 'evaluate', '--data', DATA, *UPSTREAM,
    '--weights', ONNX, '--output', ONNX_VAL, '--split', 'val', '--imgsz', IMAGE_SIZE)
for report in (PT_VAL / 'metrics.json', ONNX_VAL / 'metrics.json'):
    print(report.name, report.parent.name, report.read_text())


## 6. Optional Intel OpenVINO INT8 conversion
Off by default. Calibration uses **only training images**, never validation or test.
Colab CPU timing is not customer-device timing. Quantization can hurt recall; compare the
validation metrics and then measure the actual Intel hardware.


In [ ]:
EXPORT_OPENVINO_INT8 = False  # @param {type:"boolean"}
if EXPORT_OPENVINO_INT8:
    run('-m', 'pip', 'install', '-q', 'openvino>=2025.2,<2027', 'nncf>=2.14,<4')
    OV_OUT = OUTPUT_ROOT / (RUN_ID + '-openvino-int8')
    run('scripts/export_weapons.py', '--data', DATA, *UPSTREAM, '--weights', BEST,
        '--target', 'openvino-int8', '--output', OV_OUT, '--imgsz', IMAGE_SIZE)
    manifest = json.loads((OV_OUT / 'candidate.json').read_text())
    run('scripts/train_weapons.py', 'evaluate', '--data', DATA, *UPSTREAM,
        '--weights', OV_OUT / manifest['artifact'], '--split', 'val', '--imgsz', IMAGE_SIZE,
        '--output', OUTPUT_ROOT / (RUN_ID + '-openvino-int8-val'))


## 7. Timing and final held-out evaluation
Timing includes preprocessing/inference/postprocessing for one image through the Python adapter.
It does not establish real camera throughput or false alarms. Exported runtimes use their own
threading defaults. Re-run the benchmark on target hardware alongside the existing detector.
Enable final test evaluation only after choosing settings on validation.


In [ ]:
sample = next((DATA_ROOT / 'images/val').glob('*.jpg'))
run('scripts/benchmark_weapons.py', '--weights', BEST, ONNX, '--image', sample,
    '--imgsz', IMAGE_SIZE, '--runs', '30', '--output', TRAIN_OUT / 'cpu-latency.json')
RUN_FINAL_TEST = False  # @param {type:"boolean"}
if RUN_FINAL_TEST:
    for label, weights, device in [('pt', BEST, '0'), ('onnx', ONNX, 'cpu')]:
        run('scripts/train_weapons.py', 'evaluate', '--data', DATA, *UPSTREAM,
            '--weights', weights, '--device', device, '--split', 'test', '--imgsz', IMAGE_SIZE,
            '--output', OUTPUT_ROOT / (RUN_ID + '-' + label + '-test'))


## 8. Download the candidate, reports and conversion scripts
The results ZIP includes weights, ONNX, metrics, provenance, attribution and scripts.
It excludes dataset images. On another computer, re-download/prepare the data to evaluate
or calibrate exports. Keep attribution with the dataset and derived training records.

Additional device commands (run after unpacking the kit, with matching data paths):
- **Apple:** install `coremltools>=9,<10` and `numpy<=2.3.5` in a Mac conversion environment;
  run `scripts/export_weapons.py --target coreml-fp16` with your `--weights`, `--data`,
  `--output`, `--imgsz` and `--allow-upstream-annotations` arguments.
- **NVIDIA/Jetson:** install the TensorRT version matching the target CUDA/JetPack runtime;
  use `--target tensorrt-fp16 --device 0`. Build and benchmark on that target GPU.
  A Colab-generated TensorRT engine is not a universal deployment artifact.
- **Intel FP32:** install OpenVINO and use `--target openvino-fp32`.

See `docs/weapon-colab.md` in the ZIP for full commands and validation guidance.
A production decision needs reviewed CCTV, per-class recall and false alarms per camera-hour.


In [ ]:
import zipfile
result_zip = OUTPUT_ROOT / (RUN_ID + '-results.zip')
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in sorted(OUTPUT_ROOT.glob(RUN_ID + '*')):
        if folder.is_dir():
            for path in sorted(folder.rglob('*')):
                if path.is_file() and '.runtime' not in path.parts:
                    archive.write(path, 'runs/' + path.relative_to(OUTPUT_ROOT).as_posix())
    for name in EMBEDDED_FILES:
        archive.write(KIT / name, name)
    for name in ('dataset.yaml', 'sources.csv', 'preparation.json', 'dataset-audit.json', 'preview.jpg'):
        archive.write(DATA_ROOT / name, 'dataset-records/' + name)
    for path in (DATA_ROOT / 'attribution').iterdir():
        archive.write(path, 'dataset-records/attribution/' + path.name)
print('Results:', result_zip, 'MB:', round(result_zip.stat().st_size / 1e6, 1))
from google.colab import files
files.download(str(result_zip))
